In [1]:
import pandas as pd
import requests
import time
import os
import logging
from scrapy import Selector
from scrapy.crawler import CrawlerProcess
import scrapy
import boto3
from dotenv import load_dotenv
import numpy as np
from fake_useragent import UserAgent


load_dotenv()
AWS_KEY = os.environ["AWS_KEY"]
AWS_SECRET_KEY = os.environ["AWS_SECRET_KEY"]
BUCKET = os.environ["BUCKET"]

In [2]:
data_path = {
    "gps" : "data_gps.csv",
    "weather" : "data_weather.csv",
    "booking" : "data_booking.csv"
}

session = boto3.Session(aws_access_key_id = AWS_KEY, 
                        aws_secret_access_key = AWS_SECRET_KEY, 
                        region_name = "eu-west-3")

s3 = session.resource("s3")
data_bucket = s3.Bucket(BUCKET)

In [17]:
type(item.key)

str

In [ ]:
dl_data = []
for item in data_bucket.objects.all():
        if any(data in item.key for data in data_path.values()):
            data_bucket.download_file(item.key, f"dl_{item.key}")


s3.ObjectSummary(bucket_name='dsfsft36-bucket-jld', key='S3_data_booking.csv')
s3.ObjectSummary(bucket_name='dsfsft36-bucket-jld', key='S3_data_gps.csv')
s3.ObjectSummary(bucket_name='dsfsft36-bucket-jld', key='S3_data_weather.csv')


In [3]:
data_set = {}
for key, path in data_path.items():
    data_set[key] = pd.read_csv(f"dl_S3_{path}", index_col=0)


In [ ]:
pd.DataFrame().drop_duplicates().drop

In [4]:
for key, df in data_set.items():
    df.dropna(inplace = True)
    df.drop_duplicates(inplace = True)

    data_set[key] = df


In [5]:
data_set["booking"]

,city,name,url,review_score,description,location
0,Mont Saint Michel,Hôtel Vert,https://www.booking.com/hotel/fr/vert.fr.html?...,"8,1","3 lits (2 lits simples, 1 lit double)",Le Mont-Saint-Michel
1,Mont Saint Michel,Le Relais Saint Michel,https://www.booking.com/hotel/fr/le-relais-sai...,"8,2",Types de lits : 1 lit double ou 2 lits simples,Le Mont-Saint-Michel
2,Mont Saint Michel,Auberge Saint Pierre,https://www.booking.com/hotel/fr/auberge-saint...,"8,1",4 lits simples,Le Mont-Saint-Michel
3,Mont Saint Michel,Apparthôtel Mont Saint Michel - Résidence Fleu...,https://www.booking.com/hotel/fr/residence-fle...,"8,4","4 lits (1 lit simple, 1 lit double, 2 lits sup...",Beauvoir
4,Mont Saint Michel,Hotel Rose,https://www.booking.com/hotel/fr/gue-de-beauvo...,"8,5",Hébergement géré par un particulier,Beauvoir
...,...,...,...,...,...,...
873,La Rochelle,Vacancéole - Le Domaine du Château - La Rochel...,https://www.booking.com/hotel/fr/et-residence-...,"5,7","2 lits (1 lit double, 1 canapé-lit)",Lagord
874,La Rochelle,Première Classe La Rochelle Sud-Aytré,https://www.booking.com/hotel/fr/premiere-clas...,"6,8",2 lits simples,Aytré
875,La Rochelle,Premiere Classe La Rochelle Nord - Puilboreau,https://www.booking.com/hotel/fr/premiere-clas...,"6,3","2 lits (1 lit double, 1 lit superposé)",Puilboreau
876,La Rochelle,"hotelF1 La Rochelle Angoulins ""Rénové""",https://www.booking.com/hotel/fr/hotelf1-la-ro...,"6,9",2 lits simples,Angoulins-sur-Mer


Data cleaning

In [6]:
data_set["gps"]

,0
0,"b'[{""place_id"":267036275,""licence"":""Data \xc2\..."
1,"b'""osm_id"":117858,""lat"":""46.1597320"",""lon"":""-1..."
2,"b'6409735277294301,""addresstype"":""town"",""name""..."
3,"b'rance m\xc3\xa9tropolitaine, 17000, France"",..."
4,"b'""licence"":""Data \xc2\xa9 OpenStreetMap contr..."
5,"b'""47.7470598"",""lon"":""5.7321403"",""class"":""boun..."
6,"b'sstype"":""village"",""name"":""La Rochelle"",""disp..."
7,"b'opolitaine, 70120, France"",""boundingbox"":[""4..."
8,"b'ata \xc2\xa9 OpenStreetMap contributors, ODb..."
9,"b',""lon"":""-1.0642807"",""class"":""boundary"",""type..."


In [7]:
data_set["weather"]

,city,date,cloudiness,temp_min,temp_max,humidity,pop,rain
0,Mont Saint Michel,2025-09-17,64,15.45,17.35,83,0.00,0.00
1,Mont Saint Michel,2025-09-17,36,18.65,20.72,63,0.00,0.00
2,Mont Saint Michel,2025-09-17,0,21.83,21.83,47,0.00,0.00
3,Mont Saint Michel,2025-09-17,0,18.60,18.60,72,0.00,0.00
4,Mont Saint Michel,2025-09-17,0,16.31,16.31,88,0.00,0.00
...,...,...,...,...,...,...,...,...
1395,La Rochelle,2025-09-21,92,16.31,16.31,70,0.92,1.70
1396,La Rochelle,2025-09-21,99,14.70,14.70,71,0.60,0.38
1397,La Rochelle,2025-09-22,98,14.78,14.78,70,0.30,0.00
1398,La Rochelle,2025-09-22,93,14.20,14.20,74,1.00,1.56


Download data to RDS for storage